# Posterior sampling of rupture rates: full workflow

This notebook runs the whole workflow from the raw `cru_A.txt` / `cru_D.txt` files to the summary tables:

1. self-tests of the sampler against exact answers;
2. loading, cleaning and characterising the problem (`preliminaries.py`), with the MFD bins' uncertainty level set by `MFD_UNCERTAINTY`;
3. replicate runs (`gibbs.py`) under the spike-and-slab prior (each rupture switched off, or on with a log-uniform rate below a soft top at its bin's target rate R_j, and the fraction on in each bin estimated) and the Gaussian likelihood, with, by default, every data group scored half linearly and half in log with the same sds (`MISFIT = 'half'`, the default; `'linear'` or `'log10'` alone), every prediction kept above zero (with `MISFIT = 'linear'`, optionally a soft band on the slip rates, `SLIP_BAND_SOFT`), and optionally a balance term that keeps each data type from being predicted all low or all high (`DOWN_THE_MIDDLE`, off by default), and a soft term on the slip along the plate motion summed along cross-sections perpendicular to it (`SLIP_SUM`, on by default, on the strips `CROSS_SECTIONS` sets);
4. convergence checks (R̂ against its noise baseline, effective sample sizes, the hyperparameters, agreement between replicates; `check_convergence.py`) and how well each row block is fitted (`block_fit.py`);
5. the results (exceedance probabilities per rupture, subsection and bin totals; `make_tables.py`), exported as CSV files keyed by the original rupture and row numbers, with representative draws and figures.

**Setup.** Keep this notebook beside the `tools/` folder, which holds the scripts it runs (`cru_common.py`, `preliminaries.py`, `gibbs.py`, `check_convergence.py`, `block_fit.py`, `make_tables.py`, `select_draws.py`, `plot_rate_magnitude.py`, `plot_slip_rates.py`, `plot_paleo_rates.py`, `make_solution.py`, `plate_rate.py`, `best_fit_as_fitted.py`) and the others, for the command line. The two data files (with, if there is one, the model's OpenSHA solution folder, such as `NZSHM22_InversionSolution-211k`, whose rupture set lets the workflow write rates as OpenSHA solutions) and the outputs can live in folders of their own: set `DATA_DIR` (whose files are only read) and `RUN_DIR` (everything the workflow writes: the cleaned problem, the runs, figures and CSV tables) in the configuration cell. By default `DATA_DIR` is `~/inversions/NZSHM22_Matrices-211k/` and `RUN_DIR` a new subfolder of it named after the settings and the start time (`SLIP_SECTIONSauto_MFD_UNCERTAINTYas-given_DOWN_THE_MIDDLE_False_preset_conservative_floor-10_positive_slipsumprojall_L08.5_<unix time>`); relative paths are taken from this notebook's folder. Within `RUN_DIR`, the cleaned problem and diagnostics stay at the top, the best fit's outputs go into `nnls/` (its rate vectors and figures and, with the model's OpenSHA rupture set, `RUPTURE_SET`, the best fit's rates for OpenSHA, `nnls/rates.csv`), and the runs, tables and figures go into `spikeslab/`, each with a copy of the configuration; to keep two settings (a second floor, say), give the second its own `RUN_DIR`, as the default name does. A different `MFD_UNCERTAINTY` or `SLIP_SECTIONS` changes the cleaned problem, so it too needs its own `RUN_DIR`. Everything the notebook prints (its own messages, the scripts' output, warnings, and any error with its traceback) is also written to a log in that subfolder, `workflow_log_<date>-<time>.txt`, a new one each time the configuration cell runs; in Jupyter the log also records when each cell started and how long it took, and which figures were shown. Each step is a script, and each also keeps what it prints in a report next to its outputs: `prelim_summary.txt` (cleaning, at the top of `RUN_DIR`), `nnls/solution_report.txt` (the best fit's rates for OpenSHA), and in `spikeslab/` `<run>_log.txt` (each run), `convergence_report.txt`, `block_fit_report.txt`, `tables_report.txt`, `select_report.txt` (the representative draws), `solution_report.txt` (the posterior mean's rates for OpenSHA) and each figure's table in `.txt`. Requirements: numpy, scipy, pandas, matplotlib, and a C compiler (`xcode-select --install` on macOS) for the Gibbs kernel. Edit the configuration cell, then run all cells in order. `PRESET = 'conservative'`, the default, uses production settings (8 chains × 100,000 sweeps, 2 replicates: on an M2's four performance cores about 65 minutes per replicate at 211,000 ruptures under the default settings); `'very_conservative'` (16 chains × 300,000 sweeps, 3 replicates: about 6 h per replicate and 18 h in all at 211,000 ruptures) aims at 10,000 effective draws per rupture, so that exceedance probabilities near 0.02 carry under 10% Monte Carlo error. The lengths were raised on November 5th, 2026, for the CFM's slip rates and the paleo misfit in years, under which the ruptures switch on and off more slowly (TESTING.md, 'Run lengths, timing and storage'). `'quick'` takes minutes. See `REPORT.md` for the reasoning behind every choice.

In [ ]:
# Configuration: edit this cell, then run all cells in order.
import os, sys, json, time, subprocess, csv
import numpy as np

DATA_DIR = '~/inversions/NZSHM22_Matrices-211k/'  # folder with the input data files, which are only read (RUN_DIR below is by default a new subfolder of it); relative paths are taken from this
                                           # notebook's folder (the scripts are in its tools/)
A_FILE, D_FILE = 'cru_A.txt', 'cru_D.txt'  # input file names, inside DATA_DIR
BLOCKS = None                              # row groups by original row numbers, e.g. 'S=0-387,MFD=695-713,P=714-753'. None detects the slip-rate rows (the leading rows, up to the first
                                           # zero-forcing penalty row) and the magnitude-bin rows; any group named is checked against them
SLIP_BLOCK, BIN_BLOCK = 'S', 'MFD'         # blocks used for subsection and magnitude-bin summaries
SLIP_SECTIONS = 'auto'                     # slip-rate measurements: 'auto' (default): each run of subsections holding copies of one fault section's slip rate is one measurement (its k rows are
                                           # divided by sqrt(k)); a run stops at each parent fault's end (with the rupture set, since November 5th, 2026), so sqrt(k) counts the copies on one fault's subsections and
                                           # CFM_TEMPLATE_RHO / CFM_RATE_LINKS the rates passed between faults; rows unlike their neighbours (a geodetic model, say) are unchanged; 'none': every subsection its own
                                           # measurement (the inputs as given; a sensitivity test, in its own RUN_DIR); or a file in DATA_DIR of 'original_row section' lines (e.g. the rupture set's subsection-to-section map)
MFD_UNCERTAINTY = 'as-given'               # the MFD bins' uncertainty level, applied at cleaning; their bin-to-bin shape (the power, 0.25 in the NZSHM22 inputs) stays as cru_A/cru_D give it.
                                           # 'as-given' (default): the inputs' uncertainties (10% at M7.0-7.1, their design value), each bin its own measurement; 'two-measurements': the bins are
                                           # a projection of two measurements, the rate of M>=5 and the b-value, so together they count as two measurements, not one per bin: every bin's
                                           # uncertainty is multiplied by sqrt(bins / 2) (3 for 18 bins: 30% at M7.0-7.1); a number: the M7.0-7.1 bin's relative uncertainty, one sd (0.1
                                           # reproduces the inputs). A different setting changes the cleaned problem, so it needs its own RUN_DIR
DOWN_THE_MIDDLE = False                    # a second term in the misfit (gibbs.py --down-the-middle): each data type's relative shortfall s = sum of predictions / sum of targets - 1 (slip-rate
                                           # sections, MFD bins, paleo rows, in units of each measurement's uncertainty) costs A s^2. True: the same A for every type, so predicting a type all at
                                           # 0, or summing to twice its targets, is equally bad for each; A is the largest such weight that keeps every type at or below one extra measurement
                                           # (30-63% of each type's zero-model misfit for the NZSHM22 inputs). 'zero-model': each type's own A, its zero-model misfit, so s = -1 or +1 costs a type
                                           # what the zero model costs it in chi2 (1.6-3.4 times True's weight). False (default): the Gaussian misfit alone (with the slip-rate and paleo bands).
                                           # The sampler only: the best fit ignores it. Each setting needs its own RUN_DIR (REPORT.md, 'Balance between over- and underprediction')
SLIP_BAND_SOFT = None                      # a soft band on the slip-rate rows, with MISFIT = 'linear' (gibbs.py --slip-log-band): a Gaussian penalty on each subsection's log ratio of
                                           # prediction to target, SLIP_BAND_SOFT decades at 1 sd (1: within an order of magnitude at 1 sd, two at 2 sd), each 1/k of its section's. None
                                           # (default): not imposed. Recorded apart from chi2; about 1.5 times the time per sweep at 211,000 ruptures, and about half the effective draws
MISFIT = 'half'                            # how every data group is scored, each with its own effective sd (gibbs.py --slip-misfit, --mfd-misfit, --paleo-misfit). 'linear':
                                           # the slip rates by their split normals (the inputs' Gaussians without CFM_SLIPRATES), the MFD bins and the paleo rows by the inputs'
                                           # Gaussians in rate. 'log10': the slip rates and the MFD bins by Gaussians in log10(prediction / target) with sd log10(1 + sd / target) on
                                           # each side (sd that side's effective sd: the template weights' and MFD_UNCERTAINTY's included), the paleo rows by each site's
                                           # recurrence-interval PDF in years (a curve through its 2.275-97.725% points with its mode at 1/bestEstimate; paleo_misfit.py). 'half'
                                           # (default since 2026-11-10): half of each, the geometric mean of the two densities, so each measurement still counts once and a
                                           # prediction an order of magnitude off is unlikely, two far more so. Every prediction is kept above zero whatever the setting.
                                           # 'log10' and 'half' need the rupture set (the paleo curves); SLIP_BAND_SOFT only with 'linear'. A different setting needs its own RUN_DIR
CFM_SLIPRATES = True                       # the slip rates. False: the inputs' Gaussians (sd twice the half range, symmetric). True (default since 2026-11-05; preliminaries.py --cfm-sliprates,
                                           # with the rupture set): each subsection's slip rate a split normal in mm/yr from its parent fault's line of the NZ Community Fault Model (CFM_FILE):
                                           # mode 0.8 x SR_pref (with CFM_PREF = 'median', its median), SR_min and SR_max at 2 sd (the inputs read theirs as 1), times SLIP_SD_MULTIPLIER (3; 2, the
                                           # inputs' doubling), so each side's sd is 0.8 x 3/2 x its width (the 0.8, the assumed fraction of the slip rate the ruptures take up, scales the whole
                                           # distribution, where the inputs scale only their targets); the slip sums (SLIP_SUM) follow. A different setting needs its own RUN_DIR
CFM_FILE = '~/Library/CloudStorage/ProtonDrive-c.rollins@gns.cri.nz-folder/Inversions/Papers/CFM/NZ_CFM_v1_0.csv'
                                           # the CFM as CSV (Name, SR_pref, SR_min, SR_max), for CFM_SLIPRATES
CFM_PREF = 'mode'                          # with CFM_SLIPRATES, what SR_pref is: 'mode' (default), the most likely slip rate, the maximum-likelihood estimate (best offset / best age); 'median'
                                           # (preliminaries.py --cfm-pref median): each split normal shifted, its sds kept, so that its median is 0.8 x SR_pref (towards the short side; a fault
                                           # whose mode would then fall at or below 0 keeps SR_pref as the mode, 18 at 211,000 ruptures). A different setting needs its own RUN_DIR
CFM_ROUNDING = False                       # with CFM_SLIPRATES, the CFM's rounding: True (preliminaries.py --cfm-rounding): each fault's SR_pref, SR_min and SR_max known only to half the finest
                                           # step among them as written (1, 0.1, 0.01 mm/yr, ...; a fault's numbers taken as set at one precision), that uniform error's variance added in
                                           # quadrature to each side's sd (a median 0.5% wider at 211,000 ruptures, 76 faults over 10%); False (default): the numbers as written. A different
                                           # setting needs its own RUN_DIR
CFM_TEMPLATE_RHO = None                    # with CFM_SLIPRATES, slip rates set from one template: faults of QualCode 2-5 (rates from comparisons with nearby faults, geomorphology, geodesy, or
                                           # none) sharing an identical SR_pref / SR_min / SR_max are given correlated errors, rho (preliminaries.py --cfm-template-rho): each one's rows weighted
                                           # 1 / sqrt(1 + (n - 1) rho) for a group of n, beside the sqrt(k) of a section's subsections; 1: a group counts as one measurement; None (default): off.
                                           # A different setting needs its own RUN_DIR
CFM_RATE_LINKS = True                      # with CFM_TEMPLATE_RHO, also the faults whose rate the CFM borrowed from another's, of any QualCode (preliminaries.py --cfm-rate-links):
                                           # a comment naming the fault the rate was taken or scaled from ('SR from Lyell', '3/4 of the Hope Hurunui SR'), or an identical triple on a trace within 5 km
                                           # or on a fault of the same name (cfm_sliprates.rate_links; listed in cfm_rate_links.csv); each fault weighted 1 / sqrt(1 + m rho) for the m faults it shares
                                           # a template or a cluster of links with. True (default): on whenever CFM_TEMPLATE_RHO is; False: the templates alone. A different setting needs its own RUN_DIR
SLIP_RATE_FACTOR = 0.8                     # the assumed fraction of each fault's slip rate that the ruptures take up (preliminaries.py --slip-rate-factor, with the rupture set): every slip-rate
                                           # distribution scaled by it as a whole, its target and its sd alike (the inputs' or the CFM's; the inputs, NZSHM22, scale only their targets, an
                                           # upstream implementation error). 0.8 (default): the inputs' factor. A different setting needs its own RUN_DIR
SLIP_SD_MULTIPLIER = 3                     # each slip-rate sd this many times its uncertainty, before SLIP_RATE_FACTOR (preliminaries.py --slip-sd-multiplier, with the rupture set): 3 (default since
                                           # 2026-11-09, after a colleague's new work on the slip-rate uncertainties); 2, the inputs' doubling (sd = 2 x 0.8 x SD, or with CFM_SLIPRATES each side 2 x 0.8 x its width); 1, the uncertainties as given (for slip-rate
                                           # uncertainties estimated from the data); 2.5, the inputs exactly as given (2.5 x 0.8 = 2: NZSHM22's sd = 2 x SD, as in runs before 2026-11-05).
                                           # Subsections with an SD of 0 keep sd = the target. A different setting needs its own RUN_DIR
CROSS_SECTIONS = 'all'                     # the strips of cross-sections perpendicular to the plate motion (Beavan et al. 2002's Euler pole) that SLIP_SUM uses (gibbs.py --cross-sections;
                                           # plate_rate.py), each the subsections' mean trace length wide (8.5 km at 211,000 ruptures: 72 between the edges where the Alpine Fault goes offshore
                                           # and through the Needles-Wairarapa step-over in Cook Strait). 'all' (default): over all of New Zealand (166 strips); 'between': only between the edges.
                                           # A different setting needs its own RUN_DIR
SLIP_SUM = False                           # a soft term in the misfit (gibbs.py --slip-sum, with the rupture set, RUPTURE_SET): along every cross-section, the model's slip along the plate motion
                                           # added up (each subsection's slip vector projected on it, signed) against the targets' added up, within the uncertainty of that sum (copies of one
                                           # fault section's slip rate correlated), each subsection counting once in all, weighted by SLIP_SUM_L0, on the CROSS_SECTIONS strips (plate_rate.py).
                                           # True: added; False (default, since 2026-11-05): not. Recorded apart from chi2 (the fit check). The sampler only. A different setting needs its own
                                           # RUN_DIR
SLIP_SUM_L0 = 8.5                          # SLIP_SUM's weight: the length along the plate motion (km) that counts as one measurement. 8.5 (default): the strips' width (the mean subsection
                                           # length), so each strip's sum is one extra observation (72 between the edges, 166 over New Zealand). 120: the target sums' decorrelation length between
                                           # the edges, about 5 measurements between them; a shorter length weighs it more
RUPTURE_SET = 'auto'                       # the OpenSHA rupture set of these inputs, for writing rates as an OpenSHA solution's rates.csv (in nnls/ and spikeslab/; make_solution.py checks that
                                           # both describe the same ruptures). 'auto' (default): the one OpenSHA solution folder in DATA_DIR (e.g. NZSHM22_InversionSolution-211k), if there is
                                           # one; or that folder's name, or its ruptures/ folder, in DATA_DIR; None: no rates.csv
PRESET = 'conservative'                    # 'conservative' (default) for production runs; 'very_conservative' for the most precise results (several times longer); 'quick' to try the workflow
FLOOR = -10                                # log10 rate below which rates are negligible (-10 by default, 1e-10): spike-and-slab's slab floor (the lower end of its 'on' range), and the cutoff
                                           # below which rates count as equal in the convergence checks and the preliminaries' collective-scale check (REPORT.md, "Floors and cutoff")
REPRESENTATIVES = 100                      # representative draws for hazard (100 by default): each run keeps 1,000 whole draws (gibbs.py --keep-draws) and select_draws.py picks this many
                                           # weighted representatives, written as columns rep_01... of rate_vectors.csv; 0 turns both off
THREADS = 4                                # the M2's performance cores
SEED = None                                # the runs' random numbers: None (default): from the start time, the number at the end of the default RUN_DIR's name,
                                           # so every run is new; an integer: that seed, to repeat a run exactly; 'legacy': 101, 102, ..., as every run before November 6th,
                                           # 2026 had. Each replicate's seed is hashed from it and the replicate's number (numpy SeedSequence), so no two runs share one
RUN_SELFTESTS = False                      # exact tests of spike-and-slab and the balance term (gibbs.py --selftest current), about half a minute; worth doing once on each machine
if MISFIT not in ('linear', 'log10', 'half'):
    raise ValueError(f"MISFIT = {MISFIT!r}: use 'half' (half linear, half log), 'linear' or 'log10'")
MISFIT_MODES_ = {'linear': ('split', 'gaussian', 'gaussian'), 'log10': ('log10', 'log10', 'years'),   # as cru_common.MISFIT_MODES
                 'half': ('split+log10', 'gaussian+log10', 'gaussian+years')}
SLIP_MISFIT, MFD_MISFIT, PALEO_MISFIT = MISFIT_MODES_[MISFIT]   # each group's mode, for gibbs.py and the scripts
_T0 = int(time.time())                      # the start time: the run's name and, with SEED None, its seed
RUN_NAME = (f'SLIP_SECTIONS{SLIP_SECTIONS}_MFD_UNCERTAINTY{MFD_UNCERTAINTY}_DOWN_THE_MIDDLE_{DOWN_THE_MIDDLE}'
            f'_preset_{PRESET}_floor{FLOOR}'
            + (f'_slipsoft{SLIP_BAND_SOFT:g}' if SLIP_BAND_SOFT else '')
            + f'_misfit{MISFIT}'
            + (('_cfmmedian' if CFM_PREF == 'median' else '_cfm') + ('round' if CFM_ROUNDING else '') + (f'tmpl{CFM_TEMPLATE_RHO:g}' + ('links' if CFM_RATE_LINKS else '') if CFM_TEMPLATE_RHO is not None else '') if CFM_SLIPRATES else '') + (f'_srf{SLIP_RATE_FACTOR:g}' if SLIP_RATE_FACTOR != 0.8 else '')
            + (f'_sdx{SLIP_SD_MULTIPLIER:g}' if SLIP_SD_MULTIPLIER != 1 else '')
            + (f'_slipsumproj{CROSS_SECTIONS}_L0{SLIP_SUM_L0:g}' if SLIP_SUM else '')
            + f'_{_T0}')                             # the default RUN_DIR's name
RUN_DIR = os.path.join(DATA_DIR, RUN_NAME)  # folder for everything the workflow writes (created if missing): by default a new subfolder of DATA_DIR named after the settings and the start time,
                                           # so that each run has its own

# Run lengths. A minority of ruptures switch on and off slowly, so the runs are long; hyper_every sets how often (in
# sweeps) the fraction on in each bin is updated (cheap). Swap moves, exact exchanges of rate between near-duplicate
# ruptures in the same bin, run once per recorded draw (every thin sweeps).
PRESETS = {
    'quick':             dict(chains=4, sweeps=2400, burn=400, thin=8, hyper_every=1, replicates=2),
    # the conservative and very conservative lengths are set for the CFM's slip rates and the paleo misfit in years
    # (November 2026): tighter slip-rate constraints make the ruptures switch on and off more slowly, the bins' fractions
    # on with an autocorrelation time of about 1,600 sweeps at 211,000 ruptures (8 x 30,000 sweeps left them at 130-180
    # effective draws, against 400 needed). Cost grows with chains x sweeps (more chains save nothing), and long chains
    # waste less on burn-in (TESTING.md, 'Run lengths, timing and storage')
    'conservative':      dict(chains=8, sweeps=100000, burn=10000, thin=32, hyper_every=1, replicates=2),
    # about 450 effective draws for each bin's fraction on per replicate; on an M2's 4 performance cores about 65 min
    # per replicate at 211,000 ruptures (0.039 s per sweep)
    'very_conservative': dict(chains=16, sweeps=300000, burn=20000, thin=32, hyper_every=1, replicates=3),
    # about 2,600 effective draws for each bin's fraction on per replicate (check_convergence.py asks 4,000); about 6 h
    # per replicate and 18 h in all at 211,000 ruptures (0.074 s per sweep), and about 3 GB of files per replicate
}
if not (isinstance(DOWN_THE_MIDDLE, bool) or DOWN_THE_MIDDLE == 'zero-model'):
    raise ValueError(f"DOWN_THE_MIDDLE = {DOWN_THE_MIDDLE!r}: use True (the balance term), 'zero-model' (its per-type "
                     'weights) or False (the Gaussian misfit alone)')
if CROSS_SECTIONS not in ('all', 'between'):
    raise ValueError(f"CROSS_SECTIONS = {CROSS_SECTIONS!r}: use 'all' (all of New Zealand) or 'between' (only between "
                     'the Alpine and Cook Strait edges)')
if not isinstance(SLIP_SUM, bool):
    raise ValueError(f'SLIP_SUM = {SLIP_SUM!r}: use True (added) or False')
if isinstance(SLIP_SUM_L0, bool) or not isinstance(SLIP_SUM_L0, (int, float)) or not SLIP_SUM_L0 > 0:
    raise ValueError(f'SLIP_SUM_L0 = {SLIP_SUM_L0!r}: give a length in km above 0 (8.5 by default)')
if SLIP_BAND_SOFT is not None and (isinstance(SLIP_BAND_SOFT, bool) or not isinstance(SLIP_BAND_SOFT, (int, float))
                                   or not SLIP_BAND_SOFT > 0):
    raise ValueError(f"SLIP_BAND_SOFT = {SLIP_BAND_SOFT!r}: use None (not imposed) or a width in decades above 0 (1: an order "
                     "of magnitude at 1 sd)")
if CFM_SLIPRATES not in (True, False):
    raise ValueError(f'CFM_SLIPRATES = {CFM_SLIPRATES!r}: use True (the CFM\'s slip rates, lopsided) or False (the inputs\')')
if not (isinstance(SLIP_RATE_FACTOR, (int, float)) and SLIP_RATE_FACTOR > 0):
    raise ValueError(f'SLIP_RATE_FACTOR = {SLIP_RATE_FACTOR!r}: the fraction of the slip rates the ruptures take up, above 0 '
                     '(0.8: the inputs\' own)')
if not (isinstance(SLIP_SD_MULTIPLIER, (int, float)) and SLIP_SD_MULTIPLIER > 0):
    raise ValueError(f'SLIP_SD_MULTIPLIER = {SLIP_SD_MULTIPLIER!r}: each slip-rate sd this many times its uncertainty, above '
                     '0 (2: the inputs\' doubling; 1: as given)')
if not (SEED is None or SEED == 'legacy' or (isinstance(SEED, int) and not isinstance(SEED, bool) and SEED >= 0)):
    raise ValueError(f"SEED = {SEED!r}: use None (from the start time), an integer of 0 or more, or 'legacy'")
SEED_USED = _T0 if SEED is None else SEED          # recorded in the configuration
if CFM_TEMPLATE_RHO is not None and (isinstance(CFM_TEMPLATE_RHO, bool) or not isinstance(CFM_TEMPLATE_RHO, (int, float)) or not 0 < CFM_TEMPLATE_RHO <= 1):
    raise ValueError(f'CFM_TEMPLATE_RHO = {CFM_TEMPLATE_RHO!r}: use None (off) or a correlation above 0 and at most 1')
if not isinstance(CFM_RATE_LINKS, bool):
    raise ValueError(f'CFM_RATE_LINKS = {CFM_RATE_LINKS!r}: use True or False')
if CFM_ROUNDING not in (True, False):
    raise ValueError(f'CFM_ROUNDING = {CFM_ROUNDING!r}: use True (each fault\'s numbers known to half their step) or False')
if CFM_PREF not in ('mode', 'median'):
    raise ValueError(f"CFM_PREF = {CFM_PREF!r}: use 'mode' (SR_pref the most likely slip rate) or 'median'")
if CFM_SLIPRATES and not os.path.exists(os.path.expanduser(CFM_FILE)):
    raise FileNotFoundError(f'CFM_SLIPRATES needs the CFM as CSV: CFM_FILE = {CFM_FILE!r} not found')
if SLIP_BAND_SOFT and MISFIT != 'linear':
    raise ValueError(f"SLIP_BAND_SOFT = {SLIP_BAND_SOFT!r} with MISFIT = {MISFIT!r}: the log misfit already scores each slip-rate "
                     "prediction in log10; set SLIP_BAND_SOFT = None (or MISFIT = 'linear')")
if PRESET not in ('quick', 'conservative', 'very_conservative'):
    raise ValueError(f"PRESET = {PRESET!r}: use 'quick', 'conservative' (default) or 'very_conservative'")
if isinstance(FLOOR, bool) or not isinstance(FLOOR, (int, float)) or not -16 < FLOOR < 0:
    raise ValueError(f'FLOOR = {FLOOR!r}: give log10 of the slab floor, a negative number (-10 by default)')
if isinstance(REPRESENTATIVES, bool) or not isinstance(REPRESENTATIVES, int) or REPRESENTATIVES < 0:
    raise ValueError(f'REPRESENTATIVES = {REPRESENTATIVES!r}: give a number of draws, 0 or more (100 by default)')
if isinstance(THREADS, bool) or not isinstance(THREADS, int) or THREADS < 1:
    raise ValueError(f'THREADS = {THREADS!r}: give a number of threads, 1 or more')
P = dict(PRESETS[PRESET])
if not (MFD_UNCERTAINTY in ('as-given', 'two-measurements') or (isinstance(MFD_UNCERTAINTY, (int, float))
                                                                   and not isinstance(MFD_UNCERTAINTY, bool) and 0 < MFD_UNCERTAINTY <= 2)):
    raise ValueError(f"MFD_UNCERTAINTY = {MFD_UNCERTAINTY!r}: use 'as-given', 'two-measurements' or the M7.0-7.1 bin's "
                     'relative uncertainty as a fraction (0.3 for 30%)')
if '_NOTEBOOK_DIR' not in globals():       # this notebook's folder, remembered so this cell can be re-run after chdir
    _NOTEBOOK_DIR = os.path.abspath('.')
CODE_DIR = os.path.join(_NOTEBOOK_DIR, 'tools')   # the scripts, in tools/ beside this notebook
_resolve = lambda p: os.path.normpath(os.path.join(_NOTEBOOK_DIR, os.path.expanduser(str(p))))
DATA_DIR, RUN_DIR = _resolve(DATA_DIR), _resolve(RUN_DIR)
for _f in ('gibbs.py', 'cru_common.py', 'preliminaries.py', 'block_fit.py', 'select_draws.py',
           'plot_rate_magnitude.py', 'plot_slip_rates.py', 'plot_paleo_rates.py', 'check_convergence.py', 'make_tables.py',
           'make_solution.py', 'plot_slip_sections.py', 'plate_rate.py'):
    if not os.path.exists(os.path.join(CODE_DIR, _f)):
        raise FileNotFoundError(f'{_f} is not in tools/ beside this notebook ({CODE_DIR})')
for _f in (A_FILE, D_FILE) + (() if SLIP_SECTIONS in ('auto', 'none') else (SLIP_SECTIONS,)):
    if not os.path.exists(os.path.join(DATA_DIR, _f)):
        raise FileNotFoundError(f'{_f} is not in DATA_DIR ({DATA_DIR})')
os.makedirs(RUN_DIR, exist_ok=True)
os.chdir(RUN_DIR)                          # every output lands in RUN_DIR
sys.path.insert(0, CODE_DIR)
from cru_common import start_log, log_only  # from here on, everything printed is also written to a log in RESULTS
RESULTS = 'spikeslab'                      # this run's subfolder of RUN_DIR: its results and log
LOG_FILE = start_log(os.path.join(RUN_DIR, RESULTS,   # the log goes with the results
                                  time.strftime('workflow_log_%Y%m%d-%H%M%S.txt')),
                     header=f"cru_workflow log, started {time.strftime('%Y-%m-%d %H:%M:%S')}")
print('code:', CODE_DIR)
print('data:', DATA_DIR)
print('runs:', RUN_DIR)
print('log: ', LOG_FILE)
from cru_common import find_rupture_set
RUPTURE_DIR = (find_rupture_set(DATA_DIR) if RUPTURE_SET == 'auto'          # the OpenSHA rupture set, or None
               else find_rupture_set(os.path.join(DATA_DIR, RUPTURE_SET), search=False) if RUPTURE_SET else None)
if RUPTURE_SET not in (None, 'auto') and RUPTURE_DIR is None:
    raise FileNotFoundError(f'RUPTURE_SET = {RUPTURE_SET!r}: no OpenSHA rupture set there (indices.csv, or ruptures/indices.csv)')
SLIP_AS_GIVEN = not CFM_SLIPRATES and SLIP_RATE_FACTOR == 0.8 and SLIP_SD_MULTIPLIER == 2.5   # the inputs' slip rows untouched
if (SLIP_SUM or PALEO_MISFIT != 'gaussian' or not SLIP_AS_GIVEN) and not RUPTURE_DIR:
    raise FileNotFoundError(f'{"SLIP_SUM" if SLIP_SUM else "PALEO_MISFIT" if PALEO_MISFIT != "gaussian" else "CFM_SLIPRATES" if CFM_SLIPRATES else "SLIP_RATE_FACTOR and SLIP_SD_MULTIPLIER (2.5 with 0.8: the inputs as given, no rupture set needed)"} needs the model\'s OpenSHA rupture set '
                            '(RUPTURE_SET), for the faults\' geometry or the paleo data; none found')
print('rupture set:', RUPTURE_DIR or 'none (RUPTURE_SET): no rates are written for OpenSHA')


def rupture_dir():
    """The OpenSHA rupture set the solutions are written for, looked for when they are written, so that a solution
    folder that appears in DATA_DIR during the run still serves: RUPTURE_DIR, or, with RUPTURE_SET = 'auto' and none
    found at the start, the rupture set of the one solution folder in DATA_DIR (None if there is none, or several)."""
    if RUPTURE_DIR or RUPTURE_SET != 'auto':
        return RUPTURE_DIR
    try:
        return find_rupture_set(DATA_DIR)
    except ValueError:                            # several solution folders: name one with RUPTURE_SET
        return None


def ref_args():
    """--reference for a fit figure: the model's own OpenSHA solution's rates (its solution/rates.csv), looked for when
    the figure is drawn, so that a solution whose inversion finishes during the run is still marked: beside the rupture
    set, or, with RUPTURE_SET = 'auto' and none found at the start, in the one solution folder in DATA_DIR that has them."""
    if RUPTURE_SET is None:
        return []
    if RUPTURE_DIR:
        found = [os.path.join(os.path.dirname(RUPTURE_DIR), 'solution', 'rates.csv')]
    else:
        found = [os.path.join(DATA_DIR, d, 'solution', 'rates.csv') for d in sorted(os.listdir(DATA_DIR))]
    found = [f for f in found if os.path.isfile(f)]
    return ['--reference', found[0]] if len(found) == 1 else []


print('reference solution (looked for again when each fit figure is drawn):', (ref_args() or [None, 'none yet'])[1])
print(f'settings: {P}')

In [ ]:
def sh(*args):
    """Run one of the workflow scripts with this Python, streaming its output; stop on failure."""
    if str(args[0]).endswith('.py') and not os.path.isabs(str(args[0])):
        args = (os.path.join(CODE_DIR, args[0]),) + tuple(args[1:])      # scripts live in CODE_DIR
    cmd = [sys.executable] + [str(a) for a in args]
    print('$', ' '.join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError('command failed: ' + ' '.join(cmd[1:]))


def show(path):
    """Display a figure inline in Jupyter; elsewhere, say where it is."""
    try:
        from IPython.display import Image, display
        display(Image(filename=path))
        log_only(f'figure: {path}')
    except ImportError:
        print('figure:', path)


CONFIG = dict(preset=PRESET, seed=SEED_USED, code_dir=CODE_DIR, data_dir=DATA_DIR, run_dir=RUN_DIR, settings=P, floor=FLOOR,
              representatives=REPRESENTATIVES, a_file=A_FILE, d_file=D_FILE, blocks=BLOCKS, slip_sections=SLIP_SECTIONS,
              mfd_uncertainty=MFD_UNCERTAINTY, down_the_middle=DOWN_THE_MIDDLE, slip_band_soft=SLIP_BAND_SOFT, misfit=MISFIT, paleo_misfit=PALEO_MISFIT, mfd_misfit=MFD_MISFIT, slip_misfit=SLIP_MISFIT, cfm_sliprates=CFM_SLIPRATES, cfm_file=CFM_FILE, cfm_pref=CFM_PREF, cfm_rounding=CFM_ROUNDING, cfm_template_rho=CFM_TEMPLATE_RHO, cfm_rate_links=CFM_RATE_LINKS and CFM_TEMPLATE_RHO is not None, slip_rate_factor=SLIP_RATE_FACTOR, slip_sd_multiplier=SLIP_SD_MULTIPLIER, cross_sections=CROSS_SECTIONS, slip_sum=SLIP_SUM, slip_sum_l0=SLIP_SUM_L0, rupture_set=RUPTURE_SET, rupture_dir=RUPTURE_DIR, reference=(ref_args() or [None, None])[1],
              slip_block=SLIP_BLOCK, bin_block=BIN_BLOCK, threads=THREADS, run_selftests=RUN_SELFTESTS,
              log_file=LOG_FILE, started=time.strftime('%Y-%m-%d %H:%M'))
json.dump(CONFIG, open('workflow_config.json', 'w'), indent=1)
log_only('configuration: ' + json.dumps(CONFIG, default=str))


def results_dir():
    """This run's subfolder of RUN_DIR (RESULTS: spikeslab/): created if needed,
    with a copy of this run's configuration."""
    import shutil
    os.makedirs(RESULTS, exist_ok=True)
    if os.path.exists('workflow_config.json'):
        shutil.copy('workflow_config.json', os.path.join(RESULTS, 'workflow_config.json'))
    return RESULTS

## 1. Self-tests

The Gibbs sampler is checked against exact answers for what the default uses (`gibbs.py --selftest current`): per-rupture lower box edges (the slab floor), the soft top at each bin's target, the spike-and-slab prior, the balance term (`DOWN_THE_MIDDLE`) and swap moves between near-duplicate ruptures, each on both backends. Every line should end in `PASSED`. After any change to the sampler, run the full suite from a terminal (`python tools/gibbs.py --selftest`), which adds deep-tail truncated normals, two- and three-rate problems under flat and exponential priors, transfer moves, the retired hierarchical prior, and the priors that are now `gibbs.py` options only (the square-root slab, the linear and halfway priors).

In [ ]:
if RUN_SELFTESTS:
    sh('gibbs.py', '--selftest', 'current')

## 2. Load, clean and characterise

The penalty rows (entries 1e20; 14 in the corrected 2e4 inputs) are removed from G and d together with the parameters they apply to (405), then all-zero rows and columns are removed. The output reports the scales, the spectrum, the resolution, near-duplicate columns and the best nonnegative fit, whose fit to the data is drawn at the end of the notebook, as the posterior's is: slip rates, rates by magnitude (the MFD) and paleo rates, from the best fit's rate vectors. All of the best fit's outputs go into `nnls/` in `RUN_DIR`: its rate vectors (`nnls_vectors.npz`), the three figures (`fig_slip_rates_nnls.png`, `fig_rate_magnitude_nnls.png` and `fig_paleo_rates_nnls.png`, each with its table in `.txt`), its rates as a table with the rows of `rate_vectors.csv`, one per rupture of the inputs (`nnls_vectors.csv`: columns `rupture`, `penalised` and `best_fit`, removed ruptures 0), and, if `DATA_DIR` holds the model's OpenSHA solution folder (`RUPTURE_SET = 'auto'`, the default, finds it: `NZSHM22_InversionSolution-211k`, say; a name picks one), the best fit's rates as an OpenSHA solution's `rates.csv` (`nnls/rates.csv`, written by `make_solution.py`, with its report in `nnls/solution_report.txt`; written at the end of the notebook, as the posterior mean's are). With the model's OpenSHA solution folder in `DATA_DIR`, that solution's own predictions (its `solution/rates.csv`) are marked too, as black squares (open on the slip-rate and paleo figures), on every fit figure (looked for when each is drawn, so a solution whose inversion finishes during the run still counts), with its χ² and a `reference` column in each figure's table. Its rates on ruptures that cleaning removes are left out of those predictions, and the tables say how much (at 211,000 ruptures, 0.14% of its total rate). Before writing the rates, `make_solution.py` checks that the rupture set describes the same ruptures, every kept rupture's fault sections against its slip-rate rows and its magnitude against its bin, and stops if any differ. Cleaning also finds the row groups automatically. The slip-rate rows are the leading rows, up to the first zero-forcing penalty row (right-hand side 0, entries only on penalised ruptures), so a slip-rate row with zero slip does not end them. The magnitude-bin rows are the rows with one constant coefficient whose supports split the ruptures into disjoint groups. Groups that `BLOCKS` does not name are labelled `SLIP_BLOCK` and `BIN_BLOCK`; any group it does name must agree with the detection, or cleaning stops. Cleaning then counts each fault section's slip rate once (`SLIP_SECTIONS = 'auto'`): the inputs copy a section's slip rate, with its uncertainty, to each of its k subsections, so each run of such copies (identical targets, and identical coefficients for the ruptures they share) has its rows divided by √k. Rows unlike their neighbours, as in a geodetic slip-rate model, are unchanged; `'none'` keeps every subsection as its own measurement (a sensitivity test, in its own `RUN_DIR`), and a file of `original_row section` lines sets the sections explicitly. Cleaning reports the fit check: the χ² a draw should have if the uncertainties are right. Cleaning also checks that the slip-rate rows are plausible, printing a short profile (it stops on clear mislabelling and warns about anything suspicious, such as ruptures that appear in no slip-rate row), and checks the bin rows again on the saved problem (the per-bin priors and the bin targets rely on them). What it prints is also kept in `prelim_summary.txt`. The best fit takes about 25 s at 211,000 ruptures, where the whole of cleaning takes about a minute, and the chains start from it. The next cell prints the cleaned problem's size and fit check, and warns if `RUN_DIR` holds earlier results made on a problem cleaned otherwise.

**MFD uncertainty** (`MFD_UNCERTAINTY`, `'as-given'` by default). The magnitude bins are not independent observations: they are a projection of two, the rate of M≥5 earthquakes and the b-value, whose own uncertainties the logic tree explores through alternative inputs. The inputs give the bins relative uncertainties of 10% at M7.0–7.1, growing as R_j^−0.25 towards the rarer bins: both are design values chosen where the inputs are built, the 10% a holdover of the UCERF3 practice of weighting data types, where the slip-rate and paleo rows carry their measured uncertainties. `'as-given'` keeps them, each bin its own measurement; counted that way the 18 bins pin their total rate to ±2.8% and their slope to ±0.024 in b. `'two-measurements'` multiplies every bin's uncertainty by √(bins / 2), 3 for 18 bins, so that together they carry the information of the two measurements they come from, as each slip-rate section's copies count once: 30% at M7.0–7.1 (29–46% across the bins), and ±8.3% on the total and ±0.07 on the slope. A number sets the M7.0–7.1 bin's relative uncertainty, the other bins following (0.1 reproduces the inputs). The bin-to-bin shape always stays as the inputs give it, the bins' targets are unchanged, and the soft top's width, the bins' relative uncertainty, follows. Cleaning prints the inputs' uncertainties, the multiplier and what the bins then say about the MFD as a whole; the fit check still counts one measurement per bin. The data cannot set this level, since the bins' misfit is almost all the model's freedom; in quick runs `'two-measurements'` let the slip-rate and paleo rows raise the bins a median 8% above their targets (2% as given) and the total rate by 5%, with the other products within Monte Carlo noise (REPORT.md, "MFD uncertainty").

In [ ]:
args = ['preliminaries.py', '--A', os.path.join(DATA_DIR, A_FILE), '--d', os.path.join(DATA_DIR, D_FILE), '--out', 'cru_clean.npz',
        '--bins-block', BIN_BLOCK, '--slip-block', SLIP_BLOCK, f'--diag-floor={FLOOR}',
        '--slip-sections', SLIP_SECTIONS if SLIP_SECTIONS in ('auto', 'none') else os.path.join(DATA_DIR, SLIP_SECTIONS),
        '--mfd-uncertainty', MFD_UNCERTAINTY]
if BLOCKS:
    args += ['--blocks', BLOCKS]
if CFM_SLIPRATES:                                  # the slip-rate rows rebuilt from the CFM, as split normals
    args += ['--cfm-sliprates', os.path.expanduser(CFM_FILE)] + (['--cfm-pref', CFM_PREF] if CFM_PREF != 'mode' else [])
    args += ['--cfm-rounding'] if CFM_ROUNDING else []
    args += ['--cfm-template-rho', str(CFM_TEMPLATE_RHO)] if CFM_TEMPLATE_RHO is not None else []
    args += ['--cfm-rate-links'] if CFM_RATE_LINKS and CFM_TEMPLATE_RHO is not None else []
if not SLIP_AS_GIVEN:                              # the slip-rate factor on each whole distribution, the sds' multiplier
    args += ['--slip-rate-factor', str(SLIP_RATE_FACTOR), '--slip-sd-multiplier', str(SLIP_SD_MULTIPLIER),
             '--rupture-set', RUPTURE_DIR]
_CLEANED_BEFORE = None
if os.path.exists('cru_clean.npz'):                # the settings of a problem cleaned here before, to compare below
    from cru_common import load_problem
    _p = load_problem('cru_clean.npz')
    _CLEANED_BEFORE = (_p['slip_sections'], _p['mfd_uncertainty'], _p['cfm_sliprates'] is not None, _p['slip_rate_factor'], _p['slip_sd_multiplier'])
    del _p
sh(*args)
show('prelim.png')
for _d in ('.', 'nnls'):                           # an earlier cleaning's best-fit figures and rates would not match this
    for _f in (os.listdir(_d) if os.path.isdir(_d) else []):     # problem ('.': the layout before nnls/)
        if (_f.startswith('fig_') and '_nnls.' in _f) or (_d == 'nnls' and _f in ('rates.csv', 'solution_report.txt')):
            os.remove(os.path.join(_d, _f))
for _f in ('rates.csv', 'modules.json', 'sub_seismo_on_fault_mfds.csv'):   # the layout before nnls/rates.csv
    if os.path.exists(os.path.join('nnls', 'solution', _f)):
        os.remove(os.path.join('nnls', 'solution', _f))
for _d in ('nnls/solution', 'nnls'):              # folders left empty
    if os.path.isdir(_d) and not os.listdir(_d):
        os.rmdir(_d)

In [ ]:
from cru_common import load_problem
_p = load_problem('cru_clean.npz')                 # the cleaned problem, as the runs will read it
_before = globals().get('_CLEANED_BEFORE')         # set by the cleaning cell: the settings of a problem cleaned here before
if _before is not None and _before != (_p['slip_sections'], _p['mfd_uncertainty'], _p['cfm_sliprates'] is not None, _p['slip_rate_factor'], _p['slip_sd_multiplier']):
    _other = [k_ for k_, f_ in (('spikeslab', 'main_rep0.npz'),) if os.path.exists(os.path.join(k_, f_))]
    if _other:
        print(f'WARNING: RUN_DIR held a problem cleaned with SLIP_SECTIONS {_before[0]!r}, MFD_UNCERTAINTY '
              f"{_before[1]!r}, CFM_SLIPRATES {_before[2] if len(_before) > 2 else False}, SLIP_RATE_FACTOR {(_before[3] if len(_before) > 3 else None) or 0.8} and SLIP_SD_MULTIPLIER {(_before[4] if len(_before) > 4 else None) or 2}; the results in {', '.join(k_ + '/' for k_ in _other)} were made on it, not on this one. "
              'Give each setting its own RUN_DIR.')
FIT = _p['chi2_expect']                            # the fit check: chi2 of a model that matches the truth
print(f"cleaned problem: {_p['G'].shape[0]} rows x {_p['G'].shape[1]} ruptures; slip-rate sections {_p['slip_sections']!r}; "
      f"MFD uncertainty {_p['mfd_uncertainty']!r}" + ('; slip rates from the CFM (split normals)' if _p['cfm_sliprates'] else '') + (f"; fit check: a draw's chi2 should be near {FIT[0]:.0f} +- {FIT[1]:.0f} "
                                                       'if the uncertainties are right' if FIT is not None else ''))
if MISFIT != 'linear' and FIT is not None:          # the runs' own fit check, chi2 as fitted
    print(f'   the runs score the data with MISFIT = {MISFIT!r}, so their fit check (chi2 as fitted, which gibbs.py prints at '
          'the start of each run) has each group\'s own expected misfit in place of the Gaussian\'s 1 per measurement')
del _p


**The best fit's figures** (`nnls/fig_slip_rates_nnls.png`, `nnls/fig_rate_magnitude_nnls.png` and `nnls/fig_paleo_rates_nnls.png`, each with its table in `.txt`): the best nonnegative fit from cleaning, drawn as the posterior's are below and with the same axes. The best nonnegative fit is a Gaussian least-squares fit (every row Gaussian: the CFM's slip rates with σ_ref on both sides, the paleo rows around `bestEstimate`, the bins in rate); where the sampler scores the data otherwise (`CFM_SLIPRATES`, `MISFIT`), `best_fit_as_fitted.py` also finds the best fit under the sampler's own misfit, χ² as fitted (L-BFGS-B from the NNLS fit, about 2–3 minutes at 211,000 ruptures; `nnls/fitted_vectors.npz`, with its report), and each figure shows the two side by side. The chains then start from the best fit as fitted (`gibbs.py --init fitted`, which checks that it was made from this problem's NNLS fit), and from the NNLS fit when there is none: the start sets only how much burn-in the chains need, not where they end up. They come right after cleaning, before the runs, to show whether one data type is far off in the fit; each looks for the model's OpenSHA solution when it is drawn, so a solution whose inversion finishes later is marked by running this cell again.

In [ ]:
_rup = ['--ruptures', rupture_dir()] if rupture_dir() else []   # the fit figures in physical units, with the rupture set
_sets = ['nnls/nnls_vectors.npz:best fit (NNLS)']
if CFM_SLIPRATES or PALEO_MISFIT != 'gaussian' or MFD_MISFIT != 'gaussian' or SLIP_MISFIT != 'split':   # the sampler's misfit is not the Gaussians':
    sh('best_fit_as_fitted.py', '--problem', 'cru_clean.npz', '--bins-block', BIN_BLOCK,   # its own best fit, chi2 as fitted
       *(['--paleo-misfit', PALEO_MISFIT, '--rupture-set', RUPTURE_DIR] if PALEO_MISFIT != 'gaussian' else []),
       *(['--mfd-misfit', MFD_MISFIT] if MFD_MISFIT != 'gaussian' else []),
       *(['--slip-misfit', SLIP_MISFIT, '--slip-block', SLIP_BLOCK] if SLIP_MISFIT != 'split' else []))
    _sets.append('nnls/fitted_vectors.npz:best fit as fitted')
for _script, _fig, _more in (('plot_slip_rates.py', 'nnls/fig_slip_rates_nnls.png', _rup),     # the best fit's fit to the data
                             ('plot_rate_magnitude.py', 'nnls/fig_rate_magnitude_nnls.png', ['--bins-block', BIN_BLOCK, f'--diag-floor={FLOOR}']),
                             ('plot_paleo_rates.py', 'nnls/fig_paleo_rates_nnls.png', ['--bins-block', BIN_BLOCK] + _rup)):
    sh(_script, *_sets, '--problem', 'cru_clean.npz', '--out', _fig, '--slip-block', SLIP_BLOCK,
       *_more, *ref_args())
    show(_fig)

## 3. Default runs

**The prior (spike-and-slab):** each rupture is either off, with a rate of exactly 0, or on, with a log-uniform rate from 10^`FLOOR` (1e-10 by default, below the scale at which all ruptures together become invisible to the data) up to a top at its bin's target rate R_j. The top is soft: no rupture can carry more than its bin's true total, which the bin row gives as R_j with its own uncertainty, so the range ends at R_j with a Gaussian shoulder of that width in log rate (0.04–0.06 decades for the 2e4 inputs). Hard tops, at R_j or at F × R_j, are `gibbs.py` options (`--prior spikeslab:-10:hard` or `spikeslab:-10:F`); one at 10 × R_j left the misfits unchanged. The fraction switched on in each bin is estimated (Beta(1, 1) prior), so the data, not a floor or a depth, decide how many ruptures are active, and the prior puts no pressure on the bin totals. In quick tests on the corrected inputs, with one measurement per slip-rate section and the balance term's common weight, without the band (at a slab floor of 1e-10), χ² is 140 ± 12 against the fit check's 126 ± 16, the bin totals sit a median 2% above their targets, and about 45 ruptures carry 90% of the rate; the products barely depend on the slab floor. A minority of ruptures switch on and off slowly, so its runs are long; section 4 also checks the fractions on.

The likelihood is Gaussian, exp(−χ²/2) with the stated uncertainties, for every prior, unless `CFM_SLIPRATES` or `MISFIT` replaces a group's. `CFM_SLIPRATES = True` (the default since November 5th, 2026) takes each subsection's slip rate from its parent fault's line of the NZ Community Fault Model (`CFM_FILE`) instead of the inputs' symmetric Gaussian: a split normal in mm/yr with its mode at 0.8 × SR_pref and, on each side, an sd of 0.8 × the width to SR_min or SR_max (SR_min and SR_max read as 2 sd and multiplied by `SLIP_SD_MULTIPLIER`: with 2, as the inputs double theirs, they sit at 1 sd, and with the default 3 at 2/3 sd; `CFM_PREF = 'median'` instead shifts each split normal, its sds kept, so that its median is 0.8 × SR_pref; `CFM_ROUNDING = True` widens each side's sd in quadrature by the rounding of the fault's numbers, each known to half the finest step among them as written; `CFM_TEMPLATE_RHO` (off by default; 1: fully correlated) treats faults of quality codes 2–5 whose three numbers are identical, a shared template, as correlated, so that a group of them counts as fewer independent measurements, and with `CFM_RATE_LINKS` (on by default) also the faults of any code whose rate the CFM borrowed from another's (named in its comment, or an identical triple on an adjacent trace or a fault of the same name); the 0.8, the assumed fraction of the slip rate the ruptures take up, scales the whole distribution, where the inputs scale only their targets, so σ is 0.8 of theirs), so a slip rate with a long upper range is held loosely above and tightly below; the summed slip rates (`SLIP_SUM`) sum the modes, with the mean of the two sides as each sd ("Slip rates from the CFM" in REPORT.md). `SLIP_RATE_FACTOR` (0.8 by default, the inputs' own) is that fraction: the assumed share of each fault's slip rate the ruptures take up, scaling every slip-rate distribution as a whole, the inputs' or the CFM's, target and sd alike. `SLIP_SD_MULTIPLIER` (3 by default since November 9th, 2026, after a colleague's new work on the slip-rate uncertainties; 2, the inputs' doubling, before) scales the uncertainties: each slip-rate sd is that many times its uncertainty, before the slip-rate factor, 1 for the uncertainties as given (for slip-rate uncertainties estimated from the data), in both modes. The inputs (NZSHM22) put the factor on their targets only, an implementation error upstream in the NZSHM22 code, so with 2 their sds are 0.8 of the inputs' own, and with the default 3, 1.2 times; `SLIP_SD_MULTIPLIER = 2.5` (2.5 × 0.8 = 2) leaves the inputs exactly as given, as in runs before November 5th, 2026 ("The slip-rate factor" in REPORT.md). Every prediction is kept above zero, so at least one rupture is on in each slip-rate, paleo and MFD row, with no other limit ("Positivity" in REPORT.md; `gibbs.py --all-positive`). `SLIP_BAND`, which also allowed a factor of each slip-rate target ("The band" in REPORT.md) or no limit, was removed on November 10th, 2026 (`gibbs.py --slip-band` and `--paleo-band` remain command-line options). With `MISFIT = 'linear'`, `SLIP_BAND_SOFT` (off by default, `None`) adds a soft version: a Gaussian penalty on each prediction's log ratio to its target, that many decades at 1 sd (1: within an order of magnitude at 1 sd, two at 2 sd). It acts on the rows with weak targets, where the data would accept predictions near zero, and is kept exactly by a Metropolis step in the sampler, at about 1.5 times the time per sweep and half the effective draws ("A soft band" in REPORT.md). `MISFIT` sets how every data group is scored, each with its own effective sd. `'linear'` scores the slip rates by their split normals (the inputs' Gaussians without `CFM_SLIPRATES`) and the MFD bins and the paleo rows by the inputs' Gaussians in rate (around `bestEstimate`, with half the one-sigma range as its sd). `'log10'` scores the slip rates and the MFD bins by Gaussians in log10(prediction / target) with sd log10(1 + σ/target) on each side, σ that side's effective sd (the template weights' and `MFD_UNCERTAINTY`'s included; 0.04–0.08 decades for the bins, the soft top's width), and the paleo rows by each site's recurrence-interval PDF from the paleoseismic analysis, −2 ln of its density in years: a curve through the site's 2.275, 15.865, 84.135 and 97.725% points with its mode at 1/`bestEstimate` exactly (the rupture set's `paleo_constraint_data.json`; `paleo_misfit.py`). `'half'`, the default since November 10th, 2026, scores each by half of each: the geometric mean of the two densities, so each measurement still counts once. Near a target the two halves agree; away from it the log half grows with the number of decades off, so a prediction an order of magnitude off is unlikely and two far more so, and the paleo ranges' lopsidedness counts, where the Gaussian caps a site's misfit (a prediction of zero costs it about 8). Every prediction is kept above zero whatever the setting, each log misfit is kept exactly by Metropolis steps, and χ² as fitted counts each group as it is scored, beside the Gaussian χ² ("The slip rates in log10 as well" and "One misfit setting" in REPORT.md). `SLIP_SUM` needs the model's rupture set (`RUPTURE_SET`) and works along cross-sections perpendicular to the plate motion. `SLIP_SUM = True` (`False`, the default since November 5th, 2026, leaves it out) adds a soft term to the misfit: along every cross-section, the model's slip along the plate motion added up (each subsection's slip vector projected on the local plate motion, signed) should match the targets' added up, within that sum's uncertainty, with each subsection counting once in all; it is weighted so that `SLIP_SUM_L0` of the plate boundary counts as one measurement: 8.5 km by default, the strips' width, so each strip's sum is one extra observation (120 km, the target sums' decorrelation length between the edges, would make it about five there), and it is recorded apart from χ² ("Summed slip rates" in REPORT.md). Its strips of cross-sections are 8.5 km wide: over all of New Zealand (`CROSS_SECTIONS = 'all'`, the default) or only between two edges, where the Alpine Fault goes offshore and the Needles–Wairarapa step-over in Cook Strait (`'between'`); `plate_rate.py` builds them. Two hard constraints that were settings here until November 4th, 2026, are `gibbs.py` options only now, off unless given: an MFD that never rises where its targets do not (`--mfd-monotonic`; "A monotonic MFD" in REPORT.md), and a limit on each strip's share of the Pacific–Australia plate rate that the crustal faults take up (`--plate-rate 0.8 --rupture-set DIR`; "The plate rate" in REPORT.md).

With `DOWN_THE_MIDDLE = 'zero-model'` (off by default, `False`) the misfit gains a balance term: each data type's relative shortfall, s = sum of predictions / sum of targets − 1 (the slip-rate sections, the MFD bins and the paleo rows, in units of each measurement's uncertainty), costs A s², with each type's own A, its zero-model misfit, so a type predicted all at 0, or summing to twice its targets, pays in the balance term what the zero model costs it in χ². With `DOWN_THE_MIDDLE = True` one A serves every type instead, the largest common weight that keeps every type at or below one extra measurement: 30–63% of each type's zero-model misfit for the NZSHM22 inputs, so `'zero-model'` weighs 1.6–3.4 times as much. Each run's log gives A and each type's shortfall; the χ² of the fit check and the figures are over the data alone. In quick tests the common weight cut the posterior's mean shortfall in the slip rates by about a fifth and in the paleo rates by a third (2e4) to two-fifths (211,000 ruptures), the bins rising 1–4%, and the per-type weights (the "proposed weight") about halved the mean shortfalls in the slip and paleo rates, the bins rising up to 6% at 211,000 ruptures (REPORT.md, "Balance between over- and underprediction"). Each replicate is an independent `gibbs.py` run with its own seed, started from the best fit, with exact swap moves between near-duplicate ruptures (pairs in the same bin whose columns have cosine above 0.99, one pass per recorded draw). In quick runs at 211,000 ruptures the swap moves cost no measurable time and gave the ruptures that are often on about three times the effective draws, without moving any result (REPORT.md, "Swap moves"). The other priors tried are `gibbs.py` options, for the command line (REPORT.md): the square-root slab (`--slab sqrt`), which fits the data as the default does and so shows what depends on how the lower end of the rates is treated (at 211,000 ruptures the subsections' event rates do, coming out a median 0.62 of the default's); θ = 1 (`--prior binbox-theta:1:0`, the alternative until October 2026, which fitted no better and from about 2e5 ruptures cannot cross its boxes); and the linear, halfway and per-bin-box priors, which push the bin totals up.

In [ ]:
def gibbs_run(out, seed=1):
    """One gibbs.py run of spike-and-slab with this preset's run lengths, from the best fit, with swap moves."""
    start = 'fitted' if os.path.exists('nnls/fitted_vectors.npz') else 'nnls'   # the best fit as fitted, if made (cell above)
    args = ['gibbs.py', '--problem', 'cru_clean.npz', '--prior', f'spikeslab:{FLOOR}:soft',
            '--chains', P['chains'], '--sweeps', P['sweeps'], '--burn', P['burn'], '--thin', P['thin'],
            '--threads', THREADS, '--seed', seed, '--init', start, '--out', os.path.join(results_dir(), out)]
    args += [f'--diag-floor={FLOOR}', '--slip-block', SLIP_BLOCK, '--bins-block', BIN_BLOCK, '--no-samples']
    if REPRESENTATIVES > 0:
        args += ['--keep-draws', 1000]                  # whole draws kept for the representatives (a fixed number)
    args += ['--swap-cos', 0.99, '--hyper-every', P['hyper_every']]   # swap moves between near-duplicate ruptures
    if DOWN_THE_MIDDLE:
        args += ['--down-the-middle']                   # the balance term in the misfit
        if DOWN_THE_MIDDLE == 'zero-model':
            args += ['zero-model']                      # each type's own weight, its zero-model misfit
    args += ['--all-positive']                          # every slip-rate, paleo and MFD prediction above zero
    if SLIP_BAND_SOFT:                                  # and a soft band: SLIP_BAND_SOFT decades of the targets at 1 sd
        args += ['--slip-log-band', SLIP_BAND_SOFT]
    if PALEO_MISFIT != 'gaussian':                      # the paleo rows scored by their recurrence-interval PDFs
        args += ['--paleo-misfit', PALEO_MISFIT] + ([] if SLIP_SUM else ['--rupture-set', RUPTURE_DIR])
    if MFD_MISFIT != 'gaussian':                        # the MFD bins scored in log10, each kept above zero
        args += ['--mfd-misfit', MFD_MISFIT]
    if SLIP_MISFIT != 'split':                          # the slip rows: half the split normal, half log10 with the same sd
        args += ['--slip-misfit', SLIP_MISFIT]
    if SLIP_SUM:                                        # the summed slip rates along each cross-section, a soft term
        args += ['--slip-sum', '--slip-sum-l0', SLIP_SUM_L0]
        args += ['--cross-sections', CROSS_SECTIONS,    # their strips
                 '--rupture-set', RUPTURE_DIR]          # the faults' geometry, the ruptures' sections and slips
    sh(*args)
    return os.path.join(RESULTS, out)                   # runs live in RUN_DIR/spikeslab/


def replicate_seed(r):
    """Replicate r's seed: 101 + r with SEED 'legacy'; otherwise hashed from SEED_USED and r (numpy SeedSequence)."""
    if SEED_USED == 'legacy':
        return 101 + r
    return int(np.random.SeedSequence([SEED_USED, r]).generate_state(1)[0])


t0 = time.time()
print(f"seeds: {', '.join(str(replicate_seed(r)) for r in range(P['replicates']))} (SEED {SEED_USED!r})")
MAIN = [gibbs_run(f'main_rep{r}', seed=replicate_seed(r)) for r in range(P['replicates'])]
RUNS = MAIN
print(f'runs finished in {(time.time() - t0) / 60:.1f} min')

In [ ]:
#MAIN = RUNS = [os.path.join(RESULTS, f'main_rep{r}') for r in range(2)]

## 4. Convergence checks

For each run: χ² as fitted (each data group scored as the likelihood scores it, as `MISFIT` sets: by default half linearly and half in log, the slip rates by their split normals with `CFM_SLIPRATES`; the Gaussian χ² is printed beside it) and the total rate (effective sample sizes), and per-rupture log rates (worst R̂, effective sample sizes). The noise baseline is the worst R̂ expected over this many rates if every draw were independent, so rates above it are mixing slowly. For spike-and-slab runs, the number of ruptures on per draw and the fraction on in each bin are checked too. In the log-rate diagnostics, rates below 10^`FLOOR` (1e-10 by default, the same setting as spike-and-slab's slab floor) count as equal for every prior. Below that scale no rate, nor all of them together, can move any prediction by more than a small fraction of its uncertainty; without the clip, spike-and-slab's exact zeros would inflate R̂ with differences no prediction can see. The runs store no draws, so the per-rupture R̂ and effective draws come from the running summaries each run writes (`<run>_stream.npz`): split R̂ is exact, and effective draws are estimated from batch means. `PASS` means the targets in TESTING.md are met: enough effective draws for 95% of ruptures and for the total rate (100 and 200 at the quick preset, 1,000 and 2,000 at the conservative one, 10,000 and 20,000 at the very conservative one), and R̂ of χ² and the total rate below 1.01 (1.005 at the very conservative preset); for spike-and-slab also R̂ below 1.01 for the number of ruptures on and each bin's fraction on, with a fifth of the total rate's effective draws. Otherwise `CHECK` names the targets missed; quick-preset runs will usually say `CHECK`. The minority of ruptures that mix slowly (R̂ above 1.01, or above the noise baseline where that is higher, as in short runs; common under spike-and-slab, whose ruptures switch on and off) is reported separately, with its share of the total rate: their estimates are less precise, which the rupture table records rupture by rupture, and longer runs help them only slowly. R̂ is shown to four decimals, so the noise baseline (about 1.0003 for 16 chains of 5,000 draws) is visible. The replicate comparison should show standardized differences with sd near 1, about 0.27% beyond 3, and about the expected number beyond 4; it also tests whether the slow-mixing ruptures' stated errors hold.

`check_convergence.py` makes these checks, prints them and keeps them in `convergence_report.txt` in `spikeslab/` (with each rupture's R̂ and effective draws in `convergence.npz`, for the rupture table). Then comes each row block's fit (`block_fit.py`, also kept in `block_fit_report.txt`): the block's independent measurements (its share of the fit check), the draws' χ² over its rows with its Monte Carlo error and R̂, the χ² per measurement and its distance from the measurements in the fit check's units, and the split of that χ² into the misfit of the posterior-mean model and the posterior spread of the predictions, which is the freedom the model has in those rows. A block whose misfit is mostly spread is fitted as closely as its stated uncertainties allow, whatever they are. The MFD bins' misfit is mostly spread: it follows the width `MFD_UNCERTAINTY` gives them, so it cannot test that width, and the report says so.

In [ ]:
OUT = results_dir()                                 # reports, tables and figures go in spikeslab/
sh('check_convergence.py', *MAIN, '--problem', 'cru_clean.npz', '--preset', PRESET, f'--diag-floor={FLOOR}',
   '--out', os.path.join(OUT, 'convergence_report.txt'))
sh('block_fit.py', *MAIN, '--problem', 'cru_clean.npz', '--slip-block', SLIP_BLOCK, '--bins-block', BIN_BLOCK,
   '--out', os.path.join(OUT, 'block_fit_report.txt'))     # each row block's fit

## 5. Results

Per rupture: the posterior probability of exceeding 1e-5 and 1e-4 (near and above the single-rupture detection scale), and quantiles. Medians and lower quantiles are labelled prior-dependent: they follow the slab floor. The table also records how well each rupture mixed: `rhat_log_rate` (the worst R̂ of its log rate over the replicate runs, with rates below 10^`FLOOR` counted as equal, as in the convergence checks), `ess_log_rate` (effective draws, summed over the replicates) and `slow_mixing` (1 where R̂ exceeds 1.01, or the noise baseline if higher); `nan` marks ruptures whose every draw lies below that floor. Sums over ruptures (subsection and magnitude-bin totals) are data-determined. Everything is pooled over the replicate runs.

The subsection and bin tables also give `undetectable_share`: the share of each posterior-mean total carried by rates below their own rupture's detection scale σ/‖G_i‖ (the rate at which one rupture alone, others fixed, would move χ² by 1). A large share means the total leans on a crowd of individually invisible rates, whose number and size the prior largely sets; a small share means it rests on rates the data see one by one. In quick tests on the corrected inputs, about 11% of a typical bin's total, against 5% in the best fit.

For reference, the subsection and bin tables also give the best nonnegative fit's totals (`best_fit_total`). The best fit has no prior, so it shows the data's own tension between the slip-rate rows and the bins: for the corrected 2e4 inputs every bin sits 1–5% above its target. A posterior that departs much further from its targets than the best fit does is showing the prior's pressure.

Without stored draws, the tables come from the running summaries: exceedance probabilities, totals and shares are exact, and rupture quantiles are resolved to the histogram's 0.01-decade bins (`gibbs.py --hist-width`; rates below 1e-16 are reported as 0).

**Rate vectors** (always written, with or without draws): `rate_vectors.csv` (and `.npz`) in `spikeslab/`, pooled over the replicates. One row per rupture of the original problem, indexed as the columns of `cru_A`, so penalised ruptures are included, flagged and set to 0. Columns: the posterior mean (exact), median, mode, and the 2.275%, 15.865%, 84.135% and 97.725% values (the ±1σ and ±2σ points of a Gaussian). The median and those values come from a running histogram of log rate in 0.01-decade bins (`gibbs.py --hist-width`), interpolated within bins; in tests they agree with exact values from the draws to within one bin (typically 0.001–0.004 decades). Runs made before running summaries existed get them from their stored draws, binned the same way. The mode is defined on a log scale: 0 if a rate below 1e-16 (including exactly 0) has probability at least 0.5, otherwise the peak of the log-rate density above 1e-16, smoothed over ±0.05 decades. For weakly constrained ruptures that density is flat over the prior's range, so their mode is poorly defined; under spike-and-slab most ruptures' modes (and medians) are 0, so sum the means.

`make_tables.py` writes the tables and the rate vectors, and keeps what it prints in `tables_report.txt`. Outputs, in `RUN_DIR/spikeslab/`: the reports (`convergence_report.txt`, `block_fit_report.txt`, `tables_report.txt` and each run's `<run>_log.txt`), `ruptures_posterior.csv`, `subsections_posterior.csv` and `bins_posterior.csv`, keyed by the original rupture (column) and row numbers, the rate vectors (`rate_vectors.csv` and `.npz`), the rates-by-magnitude figure (`fig_rate_magnitude.png`, with its table in `.txt`), the slip-rate figure (`fig_slip_rates.png`, with its table in `.txt`), the paleo-rate figure (`fig_paleo_rates.png`, with its table in `.txt`), with `REPRESENTATIVES` above 0 `representatives.csv` and `select_report.txt`, with `RUPTURE_SET` the posterior mean's rates for OpenSHA (`rates.csv`, with `solution_report.txt`), and the session's log (`workflow_log_<date>-<time>.txt`).

In [ ]:
sh('make_tables.py', *MAIN, '--problem', 'cru_clean.npz', '--outdir', OUT, '--slip-block', SLIP_BLOCK,
   '--bins-block', BIN_BLOCK)                       # tables, rate vectors and tables_report.txt, in spikeslab/

**Representative draws for hazard** (`REPRESENTATIVES`, 100 by default; 0 turns this off). Each run keeps 1,000 whole posterior draws (`gibbs.py --keep-draws`); `select_draws.py` pools the replicates' kept draws and picks `REPRESENTATIVES` of them as weighted rate sets for the branches of an inversion-uncertainty level in a hazard logic tree. It chooses them to span the posterior's participation rates per subsection and magnitude bin (k-means on the principal components of their logarithms), then calibrates the weights so that the weighted mean participation rates come close to the exact posterior means. The representatives are columns `rep_01`, `rep_02`, ... of `rate_vectors.csv`, with their weights, sources, χ² and total rates in `representatives.csv`; `rate_vectors.npz` also holds the pooled kept draws. `select_report.txt` compares 10, 20, 40 and `REPRESENTATIVES` of them with the full posterior; the quick preset usually misses its tolerances. Every representative is an actual draw, so it fits the data as a draw does; the weights are quadrature weights for the posterior as a whole, and the `mean` column remains the exact source for mean hazard.

In [ ]:
if REPRESENTATIVES > 0 and os.path.exists(os.path.join(OUT, 'rate_vectors.npz')):   # weighted representative draws
    sh('select_draws.py', *MAIN, '--problem', 'cru_clean.npz', '--vectors', os.path.join(OUT, 'rate_vectors'),
       '--k', REPRESENTATIVES, '--slip-block', SLIP_BLOCK, '--bins-block', BIN_BLOCK)
else:
    print('representative draws skipped' + (' (REPRESENTATIVES = 0)' if REPRESENTATIVES <= 0 else ': no rate vectors'))

**Rates by magnitude** (`fig_rate_magnitude.png`, with its table in `fig_rate_magnitude.txt`; from the rate vectors and the predictions the runs recorded, so with or without draws): log10 of each rupture's posterior mean rate (the `mean` column of `rate_vectors.csv`) against its magnitude. A rupture's magnitude is known only to its bin, so the bin rows are counted up from 6.80–6.90 in steps of 0.1, in their original row order, and each rupture is drawn at a random magnitude within its bin to spread the points (`plot_rate_magnitude.py --lowest-bin` and `--bin-width` if the bins change). The output checks that order: the median number of subsections a rupture crosses should rise with magnitude. A black line across each bin marks its target total rate R_j, which with the soft top is also every rupture's soft upper limit; an orange dot marks the bin's total of the means, the posterior mean of the bin total. Below it, a shorter panel shows each bin's total over its target, with the band within one uncertainty of the target (at the level the likelihood gives the bins) and each of the about 3,000 kept draws' bin totals as a faint x (with `REPRESENTATIVES = 0`, the posterior 16–84% range of each bin total as a bar instead), from the predictions the runs recorded (`main_rep0.npz`, ...): on the rate axis below, a bin's spread is about as tall as its dot. Means below 10^`FLOOR`, including exactly 0, are drawn in the grey band at the bottom, and penalised ruptures are left out. The table gives, per bin, the number of ruptures, the target, the total of the means and their ratio, the posterior 16%, 50% and 84% points of the bin total, and quantiles of the log mean rates. To compare result sets side by side, give the script several rate vectors, one panel each: from `RUN_DIR`, for example, `python <code folder>/tools/plot_rate_magnitude.py theta/rate_vectors spikeslab/rate_vectors --out fig_rate_magnitude_compare.png`. With the model's OpenSHA solution folder in `DATA_DIR`, that solution's own bin totals (its `solution/rates.csv`) are marked too, as black squares, beside each bin's own total, and listed in the table.

In [ ]:
vec = os.path.join(results_dir(), 'rate_vectors.npz')
if os.path.exists(vec):                              # rates by magnitude, from the rate vectors' mean column
    fig = os.path.join(results_dir(), 'fig_rate_magnitude.png')
    sh('plot_rate_magnitude.py', vec, '--problem', 'cru_clean.npz', '--out', fig, '--bins-block', BIN_BLOCK,
       '--slip-block', SLIP_BLOCK, f'--diag-floor={FLOOR}', *ref_args())
    show(fig)
else:
    print('rates-by-magnitude figure skipped: it needs the rate vectors, written by the results cell above')

**Slip rates** (`fig_slip_rates.png`, with its tables in `fig_slip_rates.txt`; from the rate vectors and the predictions the runs recorded, so with or without draws): each fault section's slip-rate target against the mean model's prediction (G times the `mean` column of the rate vectors, which is also the posterior mean of the prediction), on a linear scale and on a log scale. A section's slip rate is one measurement, copied in the inputs to each of its subsections, so each section's mean prediction over its subsections is plotted as a dot, coloured so that neighbouring sections never share a colour (sections sharing a target stay stacked at it), with each of the about 3,000 kept draws' section means as a faint x at the section's target, so that where the draws gather shows darkest (with `REPRESENTATIVES = 0`, the posterior 16–84% range of the section mean as a bar instead), from the predictions the runs recorded. That spread is the model's, usually well under one measurement uncertainty, so a target outside it is no misfit in itself. The input rows are already divided by their uncertainties, which are not in the inputs, so both axes are in units of each section's measurement uncertainty (any √k weighting from cleaning is undone for display): the 1:1 line is unaffected, and a section mean in the grey band lies within one uncertainty of its target. A single subsection may stray further: in the likelihood it carries √k times that uncertainty. On the log scale the band reaches the bottom wherever a target is below 1, since a prediction of 0 is then within one uncertainty; means too small for that axis, including 0, sit on its bottom edge as triangles. The sections furthest from their targets in units of uncertainty are labelled with their original rows (`--label`, 3 by default), the keys of `subsections_posterior.csv`. The subtitle and the tables give the mean model's χ² over the slip-rate rows as the likelihood weights them (the χ² of the posterior mean, which is below the mean χ² over draws) and the share of sections within one and two uncertainties; the tables list every section's and every row's target, mean prediction, difference and ratio, and each section's posterior 16%, 50% and 84% points. Several rate vectors give one row of panels each. With the model's OpenSHA solution folder in `DATA_DIR`, that solution's own predictions (its `solution/rates.csv`) are marked too, as open black squares, with its χ² in the legend and a `reference` column in the tables.

In [ ]:
vec = os.path.join(results_dir(), 'rate_vectors.npz')
if os.path.exists(vec):                              # slip-rate targets against the mean model's predictions
    fig = os.path.join(results_dir(), 'fig_slip_rates.png')
    sh('plot_slip_rates.py', vec, '--problem', 'cru_clean.npz', '--out', fig, '--slip-block', SLIP_BLOCK, *ref_args(),
       *(['--ruptures', rupture_dir()] if rupture_dir() else []))     # slip rates in mm/yr, with the rupture set
    show(fig)
else:
    print('slip-rate figure skipped: it needs the rate vectors, written by the results cell above')

**Paleo rates** (`fig_paleo_rates.png`, with its table in `fig_paleo_rates.txt`; from the rate vectors and the predictions the runs recorded, so with or without draws): each paleo row's target against the mean model's prediction on log-log axes, with each of the about 3,000 kept draws' predictions as a faint x above the row's target (with `REPRESENTATIVES = 0`, the posterior 16–84% range of the prediction as a vertical bar instead), each row's marks in a colour unlike its neighbours'. As for the slip rates, both axes are in units of each row's uncertainty; on log axes that moves a point only along the 1:1 line, so its distance from the line is the ratio of prediction to target, and since a target in these units is one over its relative uncertainty, the best-measured sites lie furthest right. The shaded bands mark predictions within one and within two uncertainties of their targets, and the rows furthest from their targets are labelled with their original row numbers (`--label`, 4 by default). The paleo rows are every row in neither the slip-rate nor the magnitude-bin group (`--paleo-block` names them otherwise). The subtitle and the table give the mean model's χ² over these rows as the likelihood scores them (as `MISFIT` sets, with the rupture set: by default half the Gaussian and half −2 ln of each site's recurrence-interval PDF) and the share within one and two uncertainties; the table lists every row's target, mean prediction, posterior 16%, 50% and 84% points, difference and ratio. Several rate vectors give one panel each. With the model's OpenSHA solution folder in `DATA_DIR`, that solution's own predictions (its `solution/rates.csv`) are marked too, as open black squares, with its χ² in the legend and a `reference` column in the table.

In [ ]:
vec = os.path.join(results_dir(), 'rate_vectors.npz')
if os.path.exists(vec):                              # paleo-rate targets against the model's predictions
    fig = os.path.join(results_dir(), 'fig_paleo_rates.png')
    sh('plot_paleo_rates.py', vec, '--problem', 'cru_clean.npz', '--out', fig, '--slip-block', SLIP_BLOCK,
       '--bins-block', BIN_BLOCK, *ref_args(),
       *(['--ruptures', rupture_dir()] if rupture_dir() else []))     # rates per year and each site's ranges, with the rupture set
    show(fig)
else:
    print('paleo-rate figure skipped: it needs the rate vectors, written by the results cell above')

**Faults one by one** (`fig_slip_sections.png`, `fig_slip_sections_ruptures_1.png` and `_2.png`, and `fig_slip_sections_ruptures_by_magnitude_1.png` and `_2.png`, each set with its table in `.txt`; `plot_slip_sections.py`): with the model's rupture set (`RUPTURE_SET`), its named faults (`named_faults.json`), one column each from the most isolated to the best connected. The first figure shows each fault's posterior slip-rate prediction (each of the about 3,000 kept draws as a faint x, and the mean; with `REPRESENTATIVES = 0`, the 16–84% and 2.5–97.5% ranges and median instead) against its target, in units of its sections' uncertainties, with the share of draws in which no rupture crossing it is on and the number of other sections sharing a rupture with it. The second counts each fault's ruptures by their posterior mean rates, with each kept draw's counts of the ruptures on in it as x's; the third sums the rates by magnitude bin, the fault's rate of events per bin, each kept draw's sums as x's with the posterior mean as a circle, and a strip marked 0 whose opacity is the share of draws without an event in the bin; both show the model's OpenSHA solution's rates of the same ruptures as black outlines, and both come in two parts for readability: the first 18 faults six across, then the rest five across. Without a rupture set, 30 slip-rate sections chosen across the range of connectivity take the faults' place.

In [ ]:
vec = os.path.join(results_dir(), 'rate_vectors.npz')
if os.path.exists(vec):                              # the named faults one by one, and their ruptures
    fig = os.path.join(results_dir(), 'fig_slip_sections.png')
    _rd = rupture_dir()                              # its named_faults.json names the faults
    sh('plot_slip_sections.py', vec, '--problem', 'cru_clean.npz', '--out', fig, '--slip-block', SLIP_BLOCK,
       '--bins-block', BIN_BLOCK, *(['--ruptures', _rd] if _rd else []), *ref_args())
    for _f in ('', '_ruptures_1', '_ruptures_2', '_ruptures_by_magnitude_1', '_ruptures_by_magnitude_2'):
        if os.path.exists(fig.replace('.png', _f + '.png')):   # the rupture figures come in two parts
            show(fig.replace('.png', _f + '.png'))
else:
    print('fault-by-fault figures skipped: they need the rate vectors, written by the results cell above')

**The plate rate and the summed slip rates** (`plate_rate.txt`; `plate_rate.py`): with the model's rupture set (`RUPTURE_SET`), the share of the Pacific–Australia plate rate that the crustal faults take up in each of the `CROSS_SECTIONS` strips, its range SW of the edges, between them and NE of them: of the fault model's slip rates, the targets, the best fit, the posterior mean and the kept draws (their 2.5–97.5% range), with the strips that come closest to the whole plate rate and the faults that take most of their share (strip by strip in `plate_rate_strips.csv`); then the summed slip rates' term (`SLIP_SUM`'s), SW of the edges, between them and NE of them, for the best fit, the posterior mean and the kept draws, with the strips furthest from their target sums. Written whether or not `SLIP_SUM` is set, so that runs with and without it compare; from the command line, `plate_rate.py --limits` also counts the kept draws with every strip inside given limits.

In [ ]:
vec = os.path.join(results_dir(), 'rate_vectors.npz')
_rd = rupture_dir()                                  # the faults' traces, rakes and dips, the ruptures' slips
if not _rd:
    print("plate-rate report skipped: it needs the model's OpenSHA rupture set (RUPTURE_SET)")
elif os.path.exists(vec):
    sh('plate_rate.py', _rd, '--run', results_dir(), '--problem', 'cru_clean.npz', '--slip-block', SLIP_BLOCK,
       '--cross-sections', CROSS_SECTIONS, '--slip-sum-l0', SLIP_SUM_L0)
else:
    print('plate-rate report skipped: it needs the rate vectors, written by the results cell above')

**Rates for OpenSHA**: the best fit's (`nnls/rates.csv`) and the posterior mean's (`spikeslab/rates.csv`), each written by `make_solution.py` as an OpenSHA solution's `rates.csv`, with its report, `solution_report.txt`, beside it. Every rupture of the rupture set gets a rate, in its own numbering (the ruptures cleaning removed get 0), so each `rates.csv` has the same rows as the model's own OpenSHA solution's; `make_solution.py` first checks that the rupture set describes the same ruptures. The rupture set is looked for here, at the end, so a solution folder that appears in `DATA_DIR` during the run still serves; without one, nothing is written. From a terminal, `make_solution.py --zip` also writes a complete archive that OpenSHA loads.

In [ ]:
_rd = rupture_dir()                                  # looked for now: a solution folder that appeared during the run serves
for _vec, _col, _out, _what in (('nnls/nnls_vectors.npz', [], 'nnls', 'the best fit'),
                                (os.path.join(results_dir(), 'rate_vectors.npz'), ['--column', 'mean'], results_dir(),
                                 'the posterior mean')):
    for _f in ('rates.csv', 'solution_report.txt', 'solution/rates.csv', 'solution/modules.json',
               'solution/sub_seismo_on_fault_mfds.csv'):   # an earlier run's (solution/: the layout before rates.csv)
        if os.path.exists(os.path.join(_out, _f)):     # would not match this one
            os.remove(os.path.join(_out, _f))
    if os.path.isdir(os.path.join(_out, 'solution')) and not os.listdir(os.path.join(_out, 'solution')):
        os.rmdir(os.path.join(_out, 'solution'))
    if not _rd:
        print(f'no OpenSHA rupture set (RUPTURE_SET): {_what} is not written as rates for OpenSHA')
    elif not os.path.exists(_vec):
        print(f'{_what} is not written as rates for OpenSHA: no rate vectors ({_vec})')
    else:                                            # a rate for every rupture of the rupture set, as in its own rates.csv
        sh('make_solution.py', _vec, *_col, '--ruptures', _rd, '--problem', 'cru_clean.npz', '--out', _out,
           '--slip-block', SLIP_BLOCK, '--bins-block', BIN_BLOCK)

## 6. Open items

- A minority of ruptures that switch on and off mixes slowly: in quick runs at 211,000 ruptures about 200 (0.1%, carrying a fifth of the rate) with the swap moves, about 540 without them. Moves that pass the active role between ruptures that are not near-duplicates would help, and spike-and-slab's long run on the corrected inputs is still to do.
- The evidence of spike-and-slab, if wanted, from a few tempered runs by hand (the single-box evidence script is retired).
- An HMC cross-check of the current priors (the earlier log-space HMC script is set aside for now).
- Dispersed starts: chains started from simulated-annealing solutions or perturbed best fits (`gibbs.py --init`, from the command line), still the most important untested item.
- For the 5e5 configuration: filtered swap moves (cleaning already works on the sparse G throughout, and running summaries already allow runs without stored draws).
- Slip-rate sections are detected as runs of copies; the rupture set's subsection-to-section mapping (`SLIP_SECTIONS` as a file) would settle the sections whose target another section shares. Geodetic slip rates, which vary between subsections, are used as given; if their errors are correlated along a fault, they would need a covariance.
- The MFD bins' uncertainty level (`MFD_UNCERTAINTY`) is a modelling choice: `'as-given'` (the default) keeps the inputs' design value, each bin its own measurement; `'two-measurements'` counts the bins as the two measurements they come from, at the inputs' level and shape. A fuller treatment would give the bins a correlated covariance (the two measurements' uncertainty within a logic-tree branch, plus a model error per bin).
- Optional refinement: a similarity prior between overlapping ruptures.